In [26]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://astronomycenter.net/icop/muh47.html?l=en"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
html = response.text

soup = BeautifulSoup(html, "html.parser")

anchor = soup.find('a', {'name': 'waxing'})
if anchor is None:
    raise ValueError('No waxing anchor')

records = []
current_date = None
current_country = None

for sib in anchor.find_next_siblings():
    if sib.name == 'h1' and 'Dhul Hijjah Waning' in sib.get_text():
        break
    if sib.name == 'h2' and 'required' in sib.get('class', []):
        current_date = sib.get_text(strip=True)
        continue
    if sib.name == 'h2':
        current_country = sib.get_text(strip=True)
        continue
    if sib.name == 'div' and 'observ' in sib.get('class', []):
        text = sib.get_text(' ', strip=True)
        lower = text.lower()
        visible = 0 if 'not seen' in lower else 1 if 'seen' in lower else None
        
        if "hazy" in text:
            weather = "hazy"
        elif "cloudy" in text:
            weather = "cloudy"
        elif "clear" in text:
            weather = "clear"
        else:
            weather = "unknown"

        records.append({
            'date': current_date,
            'country': current_country,
            'visible': visible,
            'observ': text,
            'weather': weather
        })

print('total rows', len(records))
for i, r in enumerate(records, 1):
    print(i, r['date'], r['country'], r['visible'], r['weather'])


total rows 49
1 Wed 25 June 2025 Algeria 0 cloudy
2 Wed 25 June 2025 Algeria 0 hazy
3 Wed 25 June 2025 Algeria 0 hazy
4 Wed 25 June 2025 Egypt 0 clear
5 Wed 25 June 2025 Germany 0 hazy
6 Wed 25 June 2025 Iran 0 hazy
7 Wed 25 June 2025 Libya 0 clear
8 Wed 25 June 2025 Mali 0 hazy
9 Wed 25 June 2025 Morocco 0 cloudy
10 Wed 25 June 2025 Netherlands 0 hazy
11 Wed 25 June 2025 Palestine 0 cloudy
12 Wed 25 June 2025 Saudi Arabia 0 cloudy
13 Wed 25 June 2025 Tunisia 0 hazy
14 Wed 25 June 2025 United Arab Emirates 0 hazy
15 Wed 25 June 2025 United States 0 hazy
16 Wed 25 June 2025 United States 1 cloudy
17 Wed 25 June 2025 Yemen 0 hazy
18 Thu 26 June 2025 Albania 1 hazy
19 Thu 26 June 2025 Algeria 1 clear
20 Thu 26 June 2025 Algeria 1 clear
21 Thu 26 June 2025 Algeria 1 cloudy
22 Thu 26 June 2025 Australia 0 clear
23 Thu 26 June 2025 Bahrain 1 hazy
24 Thu 26 June 2025 Bangladesh 1 clear
25 Thu 26 June 2025 Bangladesh 1 cloudy
26 Thu 26 June 2025 Bosnia and Herzegovina 1 hazy
27 Thu 26 June 202

In [27]:
import pandas as pd

df = pd.DataFrame(records)

df.head()

,date,country,visible,observ,weather
0,Wed 25 June 2025,Algeria,0,1 . Time of observation: After ...,cloudy
1,Wed 25 June 2025,Algeria,0,2 . Time of observation: After ...,hazy
2,Wed 25 June 2025,Algeria,0,3 . Time of observation: After ...,hazy
3,Wed 25 June 2025,Egypt,0,1 . Time of observation: After ...,clear
4,Wed 25 June 2025,Germany,0,1 . Time of observation: After ...,hazy


In [29]:
# simpan ke excel
df.to_excel("rukyat_data.xlsx", index=False)